# 06 — Ichimoku Kinko Hyo Implementation

| Step | Content |
|------|---------|
| 1 | Data Loading & Prep |
| 2 | Ichimoku Calculation (via `ta` library) |
| 3 | NaN Handling & Data Type Verification |
| 4 | Value Spot-check |
| 5 | Ichimoku Visualisation |
| 6 | ML-safe Feature Engineering |
| 7 | Leakage Audit & Save Dataset |
| 8 | Baseline Model (No Ichimoku) |
| 9 | Enhanced Model (With Ichimoku) |
| 10 | Comparison & Conclusion |

**Dataset**: TCS_raw.csv — **Leakage note**: Senkou Spans from `ta` are
already +26-displaced; value at row t uses only data up to t-26 (ML-safe).


## Section 1 — Data Loading & Prep

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import sys, os, numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
from ta.trend import IchimokuIndicator
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, classification_report, confusion_matrix)
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), "src"))
from ichimoku import compute_ichimoku, add_ichimoku_ml_features
plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams["figure.dpi"] = 120
print("Libraries loaded successfully.")

In [ ]:
RAW_PATH  = "../data/raw/TCS_raw.csv"
SAVE_PATH = "../data/processed/TCS_ichimoku.csv"
df = pd.read_csv(RAW_PATH).iloc[1:].reset_index(drop=True)
numeric_cols = ["Open","High","Low","Close","Volume"]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").dropna(subset=numeric_cols).reset_index(drop=True)
print(f"Rows: {len(df)}")
print(f"Date range: {df['Date'].min().date()}  to  {df['Date'].max().date()}")
df.head(3)

## Section 2 — Ichimoku Calculation

| Component | Formula | Period |
|---|---|---|
| Tenkan-sen | (9-bar High + Low) / 2 | 9 |
| Kijun-sen  | (26-bar High + Low) / 2 | 26 |
| Senkou A   | (Tenkan + Kijun) / 2, +26 displaced | 26 |
| Senkou B   | (52-bar High + Low) / 2, +26 displaced | 52 |
| Chikou     | Close shifted 26 bars back | 26 |


In [ ]:
WINDOW1, WINDOW2, WINDOW3 = 9, 26, 52
df = compute_ichimoku(df, window1=WINDOW1, window2=WINDOW2, window3=WINDOW3)
ichimoku_raw_cols = ["Tenkan_sen","Kijun_sen","Senkou_Span_A","Senkou_Span_B","Chikou_Span"]
print("=== Raw Ichimoku Columns (last 5 rows) ===")
df[["Date"] + ichimoku_raw_cols].tail()

## Section 3 — NaN Handling & Data Type Verification

In [ ]:
print("=== NaN counts (before dropna) ===")
print(df[ichimoku_raw_cols].isna().sum().to_string())
df.dropna(subset=ichimoku_raw_cols, inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"\nRows after dropna: {len(df)}")
print("\n=== Data Types ===")
print(df[ichimoku_raw_cols].dtypes.to_string())
assert all(df[ichimoku_raw_cols].dtypes == "float64"), "Type error!"
print("\nAll Ichimoku columns are float64")

## Section 4 — Value Spot-check

In [ ]:
CHECK_ROW = 100
date_at_check = df.loc[CHECK_ROW, "Date"]
_raw = pd.read_csv(RAW_PATH).iloc[1:].reset_index(drop=True)
_raw[numeric_cols] = _raw[numeric_cols].apply(pd.to_numeric, errors="coerce")
_raw["Date"] = pd.to_datetime(_raw["Date"])
_raw = _raw.sort_values("Date").reset_index(drop=True)
raw_idx = _raw[_raw["Date"] == date_at_check].index[0]
w9  = _raw.loc[raw_idx - WINDOW1 + 1 : raw_idx]
w26 = _raw.loc[raw_idx - WINDOW2 + 1 : raw_idx]
manual_tenkan = (w9["High"].max()  + w9["Low"].min())  / 2
manual_kijun  = (w26["High"].max() + w26["Low"].min()) / 2
ta_tenkan = df.loc[CHECK_ROW, "Tenkan_sen"]
ta_kijun  = df.loc[CHECK_ROW, "Kijun_sen"]
print(f"Date:   {date_at_check.date()}")
print(f"Tenkan  Manual={manual_tenkan:.4f}  ta={ta_tenkan:.4f}  Match={abs(manual_tenkan-ta_tenkan)<1e-6}")
print(f"Kijun   Manual={manual_kijun:.4f}  ta={ta_kijun:.4f}   Match={abs(manual_kijun-ta_kijun)<1e-6}")

## Section 5 — Ichimoku Visualisation

Last 200 trading days. Green cloud = bullish (Span A > B), Red = bearish.

In [ ]:
plot_df = df.tail(200).copy().reset_index(drop=True)
fig, ax = plt.subplots(figsize=(16, 7))
ax.plot(plot_df.index, plot_df["Close"],       color="#2196F3", lw=1.8, label="Close Price", zorder=5)
ax.plot(plot_df.index, plot_df["Tenkan_sen"],  color="#FF6B35", lw=1.2, ls="--", label="Tenkan-sen (9)")
ax.plot(plot_df.index, plot_df["Kijun_sen"],   color="#9C27B0", lw=1.4, ls="--", label="Kijun-sen (26)")
ax.plot(plot_df.index, plot_df["Chikou_Span"], color="#4CAF50", lw=1.0, ls=":",  label="Chikou Span", alpha=0.7)
sa, sb = plot_df["Senkou_Span_A"], plot_df["Senkou_Span_B"]
ax.fill_between(plot_df.index, sa, sb, where=(sa>=sb), facecolor="#4CAF50", alpha=0.20, label="Bullish Cloud")
ax.fill_between(plot_df.index, sa, sb, where=(sa<sb),  facecolor="#F44336", alpha=0.20, label="Bearish Cloud")
ax.plot(plot_df.index, sa, color="#4CAF50", lw=0.8, alpha=0.6)
ax.plot(plot_df.index, sb, color="#F44336", lw=0.8, alpha=0.6)
ticks = plot_df.index[::20]
ax.set_xticks(ticks)
ax.set_xticklabels(plot_df["Date"].iloc[::20].dt.strftime("%b %Y"), rotation=30, ha="right", fontsize=9)
ax.set_title("TCS — Ichimoku Kinko Hyo (Last 200 Trading Days)", fontsize=14, fontweight="bold")
ax.set_ylabel("Price (INR)")
ax.legend(loc="upper left", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../data/processed/TCS_ichimoku_plot.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved.")

## Section 6 — ML-safe Feature Engineering

| Feature | Description |
|---------|-------------|
| `tk_cross` | 1 if Tenkan > Kijun |
| `price_above_cloud` | 1 if Close > max(Span A, B) |
| `price_below_cloud` | 1 if Close < min(Span A, B) |
| `cloud_bullish` | 1 if Span A > Span B |
| `cloud_thickness` | abs(Span A - Span B) |
| `chikou_vs_price` | Chikou Span - Close |


In [ ]:
df = add_ichimoku_ml_features(df)
ml_ichimoku_cols = ["tk_cross","price_above_cloud","price_below_cloud",
                    "cloud_bullish","cloud_thickness","chikou_vs_price"]
# Existing technical features
df["SMA_20"] = df["Close"].rolling(20).mean()
df["SMA_50"] = df["Close"].rolling(50).mean()
df["EMA_20"] = df["Close"].ewm(span=20, adjust=False).mean()
delta = df["Close"].diff()
df["RSI"] = 100-(100/(1+delta.clip(lower=0).rolling(14).mean()/(-delta.clip(upper=0)).rolling(14).mean()))
e12 = df["Close"].ewm(span=12,adjust=False).mean()
e26 = df["Close"].ewm(span=26,adjust=False).mean()
df["MACD"]        = e12 - e26
df["MACD_signal"] = df["MACD"].ewm(span=9,adjust=False).mean()
df["Daily_Return"]= df["Close"].pct_change()
df["Target"] = (df["Close"].shift(-1) > df["Close"]).astype(int)
df.dropna(inplace=True); df.reset_index(drop=True, inplace=True)
baseline_features = ["SMA_20","SMA_50","EMA_20","RSI","MACD","MACD_signal","Daily_Return"]
print(f"Final shape: {df.shape}")
print(df["Target"].value_counts().to_string())

## Section 7 — Leakage Audit & Save Dataset

In [ ]:
all_features = baseline_features + ml_ichimoku_cols
leaky = [f for f in all_features if f in ["Close","Target","Open","High","Low","Volume"]]
print(f"Leaky columns in feature set: {leaky if leaky else None}")
print("Senkou Spans at row t use data ending at t-26 (ML-safe).")
print("Chikou Span = Close.shift(26) = past data only (ML-safe).")
print("Target = Close.shift(-1) — future label, not used as feature.")
save_cols = ["Date"] + all_features + ichimoku_raw_cols + ["Close","Target"]
save_cols = [c for c in save_cols if c in df.columns]
df[save_cols].to_csv(SAVE_PATH, index=False)
print(f"\nDataset saved: {SAVE_PATH}  shape={df[save_cols].shape}")
df[save_cols].head(3)

## Section 8 — Baseline Model (No Ichimoku)

In [ ]:
def train_and_evaluate(X, y, label="Model"):
    X_tr,X_te,y_tr,y_te = train_test_split(X,y,test_size=0.2,shuffle=False)
    clf = RandomForestClassifier(n_estimators=200,random_state=42,n_jobs=-1)
    clf.fit(X_tr, y_tr)
    yp = clf.predict(X_te)
    m = {"label":label,
         "accuracy":accuracy_score(y_te,yp),
         "precision":precision_score(y_te,yp,zero_division=0),
         "recall":recall_score(y_te,yp,zero_division=0),
         "f1":f1_score(y_te,yp,zero_division=0),
         "model":clf,"feature_names":list(X.columns),"y_test":y_te,"y_pred":yp}
    print(f"\n{label}")
    print(f"  Acc={m['accuracy']:.4f}  Prec={m['precision']:.4f}  Rec={m['recall']:.4f}  F1={m['f1']:.4f}")
    print(classification_report(y_te, yp, target_names=["DOWN","UP"]))
    print(confusion_matrix(y_te, yp))
    return m

y = df["Target"]
baseline_metrics = train_and_evaluate(df[baseline_features], y, label="Baseline (No Ichimoku)")

## Section 9 — Enhanced Model (With Ichimoku)

In [ ]:
ichi_metrics = train_and_evaluate(df[baseline_features + ml_ichimoku_cols], y, label="Enhanced (With Ichimoku)")

## Section 10 — Comparison & Conclusion

In [ ]:
keys = ["label","accuracy","precision","recall","f1"]
comp = pd.DataFrame([{k:v for k,v in m.items() if k in keys} for m in [baseline_metrics,ichi_metrics]]).set_index("label").round(4)
comp.columns = ["Accuracy","Precision","Recall","F1"]
d = comp.iloc[1]-comp.iloc[0]; d.name = "Delta (Ichi - Base)"
print(pd.concat([comp, d.to_frame().T]).to_string())

fig, axes = plt.subplots(1, 2, figsize=(15,5))
x = np.arange(4); w = 0.35
bv = [baseline_metrics[k] for k in ["accuracy","precision","recall","f1"]]
iv = [ichi_metrics[k]     for k in ["accuracy","precision","recall","f1"]]
b1 = axes[0].bar(x-w/2, bv, w, label="Baseline",  color="#5C6BC0", alpha=0.85)
b2 = axes[0].bar(x+w/2, iv, w, label="+Ichimoku", color="#26A69A", alpha=0.85)
for b in list(b1)+list(b2):
    h=b.get_height()
    axes[0].text(b.get_x()+b.get_width()/2., h+0.002, f"{h:.3f}", ha="center", va="bottom", fontsize=8)
axes[0].set_xticks(x); axes[0].set_xticklabels(["Accuracy","Precision","Recall","F1"])
axes[0].set_ylim(0,1.10); axes[0].set_title("Metrics Comparison", fontweight="bold")
axes[0].legend(); axes[0].grid(axis="y", alpha=0.3)

fi = pd.DataFrame({"Feature":ichi_metrics["feature_names"],"Importance":ichi_metrics["model"].feature_importances_}).sort_values("Importance",ascending=True).tail(20)
colors = ["#26A69A" if f in ml_ichimoku_cols else "#5C6BC0" for f in fi["Feature"]]
axes[1].barh(fi["Feature"], fi["Importance"], color=colors, alpha=0.85)
axes[1].set_title("Top-20 Feature Importances", fontweight="bold"); axes[1].set_xlabel("Importance")
bp = mpatches.Patch(color="#5C6BC0",alpha=0.85,label="Original")
gp = mpatches.Patch(color="#26A69A",alpha=0.85,label="Ichimoku")
axes[1].legend(handles=[bp,gp],loc="lower right",fontsize=9)
plt.tight_layout()
plt.savefig("../data/processed/TCS_ichimoku_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

improved = (ichi_metrics["accuracy"]-baseline_metrics["accuracy"])>0 or (ichi_metrics["f1"]-baseline_metrics["f1"])>0
verdict = "YES - Ichimoku IMPROVED performance" if improved else "NO - Ichimoku did NOT improve performance"
print(f"\nVerdict: {verdict}")